In [1]:
import torch
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from PIL import Image

from dinosaw.utils import get_features, add_custom_font, do_2D_pca
from dinosaw.wrappers import ModelTypes, MODEL_NAMES, WRAPPER_CHECKPOINTS, get_models, get_model
from dinosaw.utils import do_2D_pca

DEVICE="cuda:0"

/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
learned_chk = f"../../models/checkpoints/{WRAPPER_CHECKPOINTS['alibi_dinov2_s_learned']}"
const_chk = f"../../models/checkpoints/{WRAPPER_CHECKPOINTS['alibi_dinov2_s']}"
alibi_learned = get_model("alibi_dinov2_s_learned", DEVICE, True, learned_chk)
alibi_const = get_model("alibi_dinov2_s", DEVICE, True, const_chk)
models = [alibi_learned, alibi_const]

2026-07-24 13:05:36 | I | factory.py                 : 152 | Building wrapper 'alibi_dinov2_s_learned' on device cuda:0
2026-07-24 13:05:36 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=False, checkpoint_path='../../models/checkpoints/ablations/alibi_coco_dv2_vits14_reg_ms_learned.pth', model_conf_path='models', stride=None, remove_pos_embed=True, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[functools.partial(<function add_alibi at 0x7fba16cd4a40>, slope_type='learned', n_reg_tokens=4, metric='euclidean', normalize=True, wrap=True, add_cls=True, jitter_mag=0.0)], dtype=torch.float32)
2026-07-24 13:05:36 | I | modifications.py           :  47 | Removed default pos. embed
2026-07-24 13:05:36 | I | factory.py                 :  80 | Loading checkpoint: ../../models/checkpoints/ablations/alibi_coco_dv2_vits14_reg_ms_learned.pth
2026-07-24 13:05:36 | I | wra

In [3]:
SF=1
img_fname="data/const_v_learned/wmg_si_c.png"


img_path = f"{img_fname}"
img = Image.open(img_path).convert('RGB')
img = img.resize((int(SF * img.width), int(SF * img.height)), Image.LANCZOS)

feats_reduced = {channel_group: {} for channel_group in range(3)}

for i, model in enumerate(models):
    feats = get_features(model, img)
    for channel_group in range(3):
        feats_reduced[channel_group][i] = do_2D_pca(feats, (channel_group+1)*3, pre_norm="std", post_norm='minmax')[:, :, channel_group*3:channel_group*3+3]

2026-07-24 13:05:37 | I | wrapper.py                 :  92 | Processing image, size: [1439, 825]


2026-07-24 13:05:37 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,812,1428] -> f: [1,384,58,102]
2026-07-24 13:05:37 | I | wrapper.py                 :  92 | Processing image, size: [1439, 825]
2026-07-24 13:05:38 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,812,1428] -> f: [1,384,58,102]


In [11]:
def hide_axis(ax):
    ax.set_xticks([])
    ax.set_yticks([])

In [15]:
plt.style.use("thesis.mplstyle")
plt.rcParams['text.usetex'] = False
add_custom_font('resources/fonts', 'Grotesk')


H,W = 2 * 2.3, 7
NROWS, NCOLS = 3,3
fig = plt.figure(figsize=(W,H))
gs = GridSpec(NROWS, NCOLS, figure=fig)

img_ax = fig.add_subplot(gs[0, 1])
img_ax.imshow(img)
hide_axis(img_ax)



for j, _ in enumerate(models):
    for i, channel_group in enumerate(list(range(3))):
        ax = fig.add_subplot(gs[j + 1, i])
        ax.imshow(feats_reduced[channel_group][j])
        ax.set_yticks([])
        ax.set_xticks([])
        if j==0:
            ax.set_title(f"Components {channel_group*3} to {channel_group*3+2}")
        if i==0:
            if j==1:
                ax.set_ylabel(r"Constant $m$", weight="bold")
            else:
                ax.set_ylabel(r"Learned $m$",)

SAVE = True
if SAVE:
    plt.savefig('saved/S7.pdf', dpi=300, bbox_inches='tight')
    plt.close()

findfont: Failed to find font weight normal, now using 300.
findfont: Failed to find font weight normal, now using 300.
